# 06 - Fraud Detection
Simple RULE-BASED fraud scoring (not machine learning) on top of clean Silver data.

Rules:
- amount_rule (+40): amount > 200,000
- velocity_rule (+30): same sender has >=5 transactions within a 10-minute window
- suspicious_ip_rule (+30): IP in a known-bad range, OR same IP used by >=3 distinct senders within an hour

score >= 60 -> HIGH, 30-59 -> MEDIUM, <30 -> LOW

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "sentinel_catalog.sentinel_schema.silver_transactions"
GOLD_FRAUD_TABLE = "sentinel_catalog.sentinel_schema.gold_fraud_transactions"

SUSPICIOUS_IP_PREFIXES = ["185.220.", "45.153.", "89.248."]

silver_df = spark.table(SILVER_TABLE)

### Rule 1: amount_rule

In [0]:
scored_df = silver_df.withColumn(
    "amount_rule_points",
    F.when(F.col("amount") > 200000, F.lit(40)).otherwise(F.lit(0))
)

### Rule 2: velocity_rule (window function over a 10-minute range per sender)

In [0]:
sender_window = (
    Window.partitionBy("sender_upi_id")
    .orderBy(F.col("event_timestamp").cast("long"))
    .rangeBetween(-600, 0)  # 600 seconds = 10 minutes
)

scored_df = scored_df.withColumn(
    "txns_in_10min", F.count("transaction_id").over(sender_window)
).withColumn(
    "velocity_rule_points",
    F.when(F.col("txns_in_10min") >= 5, F.lit(30)).otherwise(F.lit(0))
)

### Rule 3: suspicious_ip_rule

In [0]:
ip_prefix_condition = F.lit(False)
for prefix in SUSPICIOUS_IP_PREFIXES:
    ip_prefix_condition = ip_prefix_condition | F.col("ip_address").startswith(prefix)

ip_window = Window.partitionBy("ip_address").orderBy(F.col("event_timestamp").cast("long")).rangeBetween(-3600, 0)

scored_df = scored_df.withColumn(
    "distinct_senders_same_ip_1hr",
    F.approx_count_distinct("sender_upi_id").over(ip_window)
).withColumn(
    "suspicious_ip_rule_points",
    F.when(ip_prefix_condition | (F.col("distinct_senders_same_ip_1hr") >= 3), F.lit(30)).otherwise(F.lit(0))
)

### Combine into fraud_score and classify

In [0]:
fraud_df = (
    scored_df
    .withColumn(
        "fraud_score",
        F.col("amount_rule_points") + F.col("velocity_rule_points") + F.col("suspicious_ip_rule_points")
    )
    .withColumn(
        "fraud_level",
        F.when(F.col("fraud_score") >= 60, F.lit("HIGH"))
         .when(F.col("fraud_score") >= 30, F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )
    .select(
        "transaction_id", "event_timestamp", "sender_upi_id", "receiver_upi_id",
        "bank_name", "amount", "status", "ip_address",
        "amount_rule_points", "velocity_rule_points", "suspicious_ip_rule_points",
        "fraud_score", "fraud_level"
    )
)

fraud_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_FRAUD_TABLE)

### Verify

In [0]:
SELECT
    fraud_level,
    COUNT(*) AS txns,
    ROUND(AVG(fraud_score), 1) AS avg_score
FROM GOLD_FRAUD_TABLE
GROUP BY fraud_level
ORDER BY
    CASE fraud_level
        WHEN 'HIGH' THEN 1
        WHEN 'MEDIUM' THEN 2
        WHEN 'LOW' THEN 3
        ELSE 4
    END;

Total and null check

In [0]:
display(spark.sql(f"""
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT transaction_id) AS unique_transactions,
    SUM(CASE WHEN fraud_score IS NULL THEN 1 ELSE 0 END) AS null_scores,
    SUM(CASE WHEN fraud_level IS NULL THEN 1 ELSE 0 END) AS null_levels
FROM {GOLD_FRAUD_TABLE}
"""))

total_transactions,unique_transactions,null_scores,null_levels
14580,14580,0,0


Fraud-level distribution

In [0]:
display(spark.sql(f"""
SELECT
    fraud_level,
    MIN(fraud_score) AS min_score,
    MAX(fraud_score) AS max_score,
    COUNT(*) AS transaction_count
FROM {GOLD_FRAUD_TABLE}
GROUP BY fraud_level
ORDER BY min_score
"""))

fraud_level,min_score,max_score,transaction_count
LOW,0,0,13811
MEDIUM,30,30,311
HIGH,60,70,458


In [0]:
display(spark.sql(f"""
SELECT
    fraud_score,
    fraud_level,
    COUNT(*) AS cnt
FROM {GOLD_FRAUD_TABLE}
GROUP BY fraud_score, fraud_level
ORDER BY cnt DESC
LIMIT 20
"""))

fraud_score,fraud_level,cnt
0,LOW,13811
70,HIGH,428
30,MEDIUM,311
60,HIGH,30


In [0]:
dashboard_df = spark.sql(f"""
SELECT
    transaction_id,
    bank_name,
    amount,
    status,
    event_timestamp,
    fraud_score,
    fraud_level,
    amount_rule_points,
    velocity_rule_points,
    suspicious_ip_rule_points
FROM {GOLD_FRAUD_TABLE}
""")

display(dashboard_df.limit(20))

transaction_id,bank_name,amount,status,event_timestamp,fraud_score,fraud_level,amount_rule_points,velocity_rule_points,suspicious_ip_rule_points
ee4e50ed-d10e-42de-a49a-329eba166ce8,AXIS,49224.09,PENDING,2026-08-02T03:54:03.000Z,0,LOW,0,0,0
b723bf57-ce0f-4b05-a63f-642866a045ff,PNB,14288.32,TIMEOUT,2026-08-01T22:58:22.000Z,0,LOW,0,0,0
cd20bcdd-39c3-47c7-b5f3-1f3d45edd663,HDFC,11564.7,TIMEOUT,2026-08-01T12:02:22.000Z,0,LOW,0,0,0
76192b80-45a8-4863-b50a-6ce242029142,KOTAK,38045.77,TIMEOUT,2026-08-01T18:53:38.000Z,0,LOW,0,0,0
822d9ef9-1260-47f4-a5f8-d3bb152ab0de,PNB,41756.67,SUCCESS,2026-08-01T11:08:04.000Z,0,LOW,0,0,0
a7a58b0a-408e-4ff3-a197-d69b108436ea,HDFC,1323.42,FAILED,2026-08-01T14:55:21.000Z,0,LOW,0,0,0
266e722d-7ff0-4cda-bb9d-cd2a99ce5801,KOTAK,11791.77,SUCCESS,2026-08-01T22:03:38.000Z,0,LOW,0,0,0
4a166eff-b1ac-4824-8cfd-e8f9dab506a5,SBI,22050.9,TIMEOUT,2026-08-01T17:45:34.000Z,0,LOW,0,0,0
21b34a46-9ed1-48fc-8f10-18dab55b951c,AXIS,6446.36,TIMEOUT,2026-08-02T00:23:46.000Z,0,LOW,0,0,0
0dc3a802-732f-4211-85dc-0684db9d9fdb,YES_BANK,37187.91,FAILED,2026-08-01T11:46:37.000Z,0,LOW,0,0,0


Total transactions

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS total_transactions
FROM {GOLD_FRAUD_TABLE}
"""))

total_transactions
14580


High-risk transactions

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS high_risk_transactions
FROM {GOLD_FRAUD_TABLE}
WHERE fraud_level = 'HIGH'
"""))

high_risk_transactions
458


Medium-risk

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS medium_risk_transactions
FROM {GOLD_FRAUD_TABLE}
WHERE fraud_level = 'MEDIUM'
"""))

medium_risk_transactions
311


Average fraud score

In [0]:
display(spark.sql(f"""
SELECT ROUND(AVG(fraud_score), 2) AS average_fraud_score
FROM {GOLD_FRAUD_TABLE}
"""))

average_fraud_score
2.82


Step 1 — Create the dashboard KPI queries

In [0]:
display(spark.sql(f"""
SELECT
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN fraud_level = 'HIGH' THEN 1 ELSE 0 END) AS high_risk_transactions,
    SUM(CASE WHEN fraud_level = 'MEDIUM' THEN 1 ELSE 0 END) AS medium_risk_transactions,
    SUM(CASE WHEN fraud_level = 'LOW' THEN 1 ELSE 0 END) AS low_risk_transactions,
    ROUND(AVG(fraud_score), 2) AS average_fraud_score
FROM {GOLD_FRAUD_TABLE}
"""))

total_transactions,high_risk_transactions,medium_risk_transactions,low_risk_transactions,average_fraud_score
14580,458,311,13811,2.82


Step 2 — Create Fraud Level Distribution

In [0]:
display(spark.sql(f"""
SELECT
    fraud_level,
    COUNT(*) AS transaction_count
FROM {GOLD_FRAUD_TABLE}
GROUP BY fraud_level
ORDER BY transaction_count DESC
"""))

fraud_level,transaction_count
LOW,13811
HIGH,458
MEDIUM,311


Step 3 — Create Bank-wise Fraud Analysis

In [0]:
display(spark.sql(f"""
SELECT
    bank_name,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN fraud_level = 'HIGH' THEN 1 ELSE 0 END) AS high_risk_transactions,
    ROUND(AVG(fraud_score), 2) AS average_fraud_score
FROM {GOLD_FRAUD_TABLE}
GROUP BY bank_name
ORDER BY high_risk_transactions DESC
"""))

bank_name,total_transactions,high_risk_transactions,average_fraud_score
YES_BANK,2115,76,3.27
SBI,2065,68,2.89
ICICI,2096,67,2.77
KOTAK,2027,65,2.82
AXIS,2098,64,2.58
PNB,2099,60,2.66
HDFC,2080,58,2.74


Step 4 — Create Fraud Trend

In [0]:
display(spark.sql(f"""
SELECT
    DATE(event_timestamp) AS transaction_date,
    COUNT(*) AS total_transactions,
    SUM(CASE WHEN fraud_level = 'HIGH' THEN 1 ELSE 0 END) AS high_risk_transactions
FROM {GOLD_FRAUD_TABLE}
GROUP BY DATE(event_timestamp)
ORDER BY transaction_date
"""))

transaction_date,total_transactions,high_risk_transactions
2026-08-01,11657,366
2026-08-02,2923,92


Step 5 — Create Status vs Fraud Analysis

In [0]:
display(spark.sql(f"""
SELECT
    status,
    fraud_level,
    COUNT(*) AS transaction_count
FROM {GOLD_FRAUD_TABLE}
GROUP BY status, fraud_level
ORDER BY status, fraud_level
"""))

status,fraud_level,transaction_count
FAILED,HIGH,108
FAILED,LOW,3357
FAILED,MEDIUM,72
PENDING,HIGH,123
PENDING,LOW,3233
PENDING,MEDIUM,76
SUCCESS,HIGH,111
SUCCESS,LOW,3457
SUCCESS,MEDIUM,78
TIMEOUT,HIGH,116


Step 6 — Create High-Risk Transaction Table

In [0]:
display(spark.sql(f"""
SELECT
    transaction_id,
    bank_name,
    amount,
    status,
    event_timestamp,
    fraud_score,
    fraud_level,
    amount_rule_points,
    velocity_rule_points,
    suspicious_ip_rule_points
FROM {GOLD_FRAUD_TABLE}
WHERE fraud_level = 'HIGH'
ORDER BY fraud_score DESC, event_timestamp DESC
LIMIT 100
"""))

transaction_id,bank_name,amount,status,event_timestamp,fraud_score,fraud_level,amount_rule_points,velocity_rule_points,suspicious_ip_rule_points
11af06df-b56d-4fa4-99ea-8e25816dfafd,AXIS,310454.79,SUCCESS,2026-08-02T05:58:05.000Z,70,HIGH,40,0,30
439c11ab-1114-450d-89b0-2294e95f60b4,KOTAK,463436.28,FAILED,2026-08-02T05:49:18.000Z,70,HIGH,40,0,30
8ba24080-5f24-4056-9395-92d11bef428d,SBI,378818.44,PENDING,2026-08-02T05:41:22.000Z,70,HIGH,40,0,30
640c2148-f6ae-468f-a44c-1d187761395f,AXIS,466633.68,PENDING,2026-08-02T05:35:36.000Z,70,HIGH,40,0,30
4de48f69-be9b-4086-934a-f8d8cf6102ed,HDFC,469190.38,PENDING,2026-08-02T05:34:13.000Z,70,HIGH,40,0,30
e4df3003-ba88-44cd-95ab-c04c645680b7,SBI,242605.73,FAILED,2026-08-02T05:31:48.000Z,70,HIGH,40,0,30
81923755-efa3-40c7-8ab7-0e6e6e5169ef,AXIS,208160.53,TIMEOUT,2026-08-02T05:23:14.000Z,70,HIGH,40,0,30
8846de98-ce15-4ffe-ba85-07d1a49b8eab,ICICI,394900.92,TIMEOUT,2026-08-02T05:10:40.000Z,70,HIGH,40,0,30
d6cd3f5f-c301-4ffa-b38f-c428321d2dc3,AXIS,418764.47,SUCCESS,2026-08-02T05:03:51.000Z,70,HIGH,40,0,30
d2ffd615-21f2-4dc0-9806-4800786debd2,HDFC,236134.26,PENDING,2026-08-02T05:02:50.000Z,70,HIGH,40,0,30
